# EnzAIme — Full Training & Evaluation (Kaggle GPU)

End-to-end run of the machine-learning pipeline:
1. clean dataset + explainable-missingness flags (`data_quality.py`)
2. transparent training scenarios + derived compatibility labels (`label_generation.py`)
3. frozen **ESM-2 650M** embeddings, mean pooled (BOS/EOS excluded) (`esm_embedder.py`)
4. baselines (ridge/RF/GBM) + small MLP on **group split by enzyme** (`model.py`, `train_kaggle.py`)
5. explainable, rule-based inference + mutation-candidate prep

**Scientific honesty:** all labels are *derived compatibility scores*, never experimental degradation efficiency; enzyme-group splits prevent leakage; missing metadata is flagged and lowers confidence instead of being invented.

## 0. Setup
Locate the project (either mounted `/kaggle/input` dataset or `/kaggle/working` checkout) and install missing packages.

In [ ]:
import sys, importlib.util, shutil, subprocess, os
from pathlib import Path

def find_project():
    for root in [Path('/kaggle/input/enzaime'),
                 Path('/kaggle/working/enzaime'),
                 Path('/kaggle/working'), Path('..'), Path('.')]:
        if (root / 'src' / 'train_kaggle.py').exists():
            return root.resolve()
    raise FileNotFoundError('project not found — mount the EnzAIme repo as /kaggle/input/enzaime')

ROOT = find_project()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print('project root:', ROOT)

def ensure(mod):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', mod])
for m in ['matplotlib', 'tabulate']:
    ensure(m)
print('env ok · GPU:', end=' ')
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Data quality + recovery plan
Cleans the master dataset, builds `has_*` missingness flags, records a **no-download** recovery log and confident evidence metadata.

In [ ]:
import data_quality
qc = data_quality.run_data_quality(
    ROOT / 'data' / 'processed' / 'master_enzymes.csv',
    ROOT / 'data' / 'processed', ROOT / 'reports')
print(qc)

## 2. Training scenarios + transparent labels
Creates the conservative scenario grid (pH × temp × salinity) per enzyme and derives `derived_compatibility_label` with provenance-aware confidence.

In [ ]:
import label_generation
df = label_generation.generate_scenarios(
    ROOT / 'data' / 'processed' / 'master_enzymes_clean.csv',
    ROOT / 'data' / 'processed' / 'training_scenarios.csv')
print(f'scenarios: {len(df)} rows · label range {df["derived_compatibility_label"].min():.3f}..{df["derived_compatibility_label"].max():.3f}')

## 3. Frozen ESM-2 (650M) embeddings
Downloads `facebook/esm2_t33_650M_UR50D` (network needed, GPU used for batch), mean-pools hidden states excluding BOS/EOS, caches per-sequence vectors, writes `esm_embeddings.npy` (N×1280) aligned with `esm_embedding_index.csv`.

In [ ]:
import esm_embedder
res = esm_embedder.run_embedding(
    ROOT / 'data' / 'processed' / 'master_enzymes_valid_sequences.csv',
    ROOT / 'data' / 'processed',
    model_name='facebook/esm2_t33_650M_UR50D',
    batch_size=4, force=False, download=True)
print('embedded enzymes:', res.get('n_embedded') if 'n_embedded' in res else res.get('embedded', '?'))
import numpy as np; print('shape:', np.load(ROOT / 'data' / 'processed' / 'esm_embeddings.npy').shape)

## 4. Baselines + small MLP (group-split, no leakage)
Fits ridge/RF/GBM on env features and an MLP on frozen embeddings + env features, evaluated on an enzyme-group holdout. Saves artifacts + reports. `metrics.json` states explicitly that metrics measure agreement with the derived label.

In [ ]:
from train_kaggle import main as train_main
raise SystemExit(train_main(['--esm-model', 'facebook/esm2_t33_650M_UR50D', '--download']))

In [ ]:
import json
m = json.load(open(ROOT / 'artifacts' / 'metrics.json'))
print('best_model:', m['best_model'])
import pandas as pd
print(pd.DataFrame(m['model_comparison']).to_string(index=False))

## 5. Explainable inference + mutation prep
Rule-based baseline inference (always available) is used when the best trained model is a simple baseline; the ML recommender is used when the MLP wins. Mutation candidates are computational-only.

In [ ]:
from inference import recommend as rec
out = rec({'pollutant_type': 'PET', 'ph': 8.0, 'temperature_c': 35, 'salinity': 0.5}, use_ml=True)
print('mode:', out['mode'])
for r in out['recommendations'][:4]:
    print(r['enzyme_id'], r['suitability_score'], '|', r['explanation'][:120])

In [ ]:
import mutation_prep
cands = mutation_prep.generate_mutation_candidates(
    ROOT / 'data' / 'processed' / 'master_enzymes_valid_sequences.csv',
    ROOT / 'data' / 'processed' / 'esm_embeddings.npy',
    ROOT / 'data' / 'processed' / 'esm_embedding_index.csv',
    ROOT / 'artifacts',
    ROOT / 'data' / 'processed' / 'mutation_candidates.csv',
    ROOT / 'reports' / 'mutation_prep_summary.json')
print('mutation candidates:', len(cands))

## 6. Done
Artifacts: `artifacts/{best_model.pt, scaler.pkl, feature_config.json, metrics.json, model.pt, encoders.pkl, config.json}` — the last three keep the backend `ModelService` working unchanged. Reports: `reports/{training_report.md, model_comparison.csv, prediction_examples.csv, …}`.